In [1]:
import pandas as pd

In [6]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Aysel", "Orxan", "Leyla", "Murad"],
    "city": ["Baku", "Ganja", "Baku", "Sumqayit"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "customer_id": [1, 1, 2, 3, 3, 1],
    "amount": [250, 400, 150, 900, 300, 350],
    "order_date": pd.to_datetime(["2026-01-10", "2026-02-05", "2026-02-20",
                                  "2026-03-01", "2026-03-15", "2026-03-20"])
})
print(customers,"\nn","\n", orders)

   customer_id   name      city
0            1  Aysel      Baku
1            2  Orxan     Ganja
2            3  Leyla      Baku
3            4  Murad  Sumqayit 
n 
    order_id  customer_id  amount order_date
0       101            1     250 2026-01-10
1       102            1     400 2026-02-05
2       103            2     150 2026-02-20
3       104            3     900 2026-03-01
4       105            3     300 2026-03-15
5       106            1     350 2026-03-20


In [12]:
customers[customers['city']=="Baku"]

,customer_id,name,city
0,1,Aysel,Baku
1,2,Orxan,Baku
2,3,Leyla,Baku
3,4,Murad,Baku


In [14]:
customers[['name','city']]

,name,city
0,Aysel,Baku
1,Orxan,Baku
2,Leyla,Baku
3,Murad,Baku


In [15]:
orders.groupby("customer_id")["amount"].sum().reset_index()

,customer_id,amount
0,1,1000
1,2,150
2,3,1200


In [16]:
customers.merge(orders, on="customer_id",how="left")

,customer_id,name,city,order_id,amount,order_date
0,1,Aysel,Baku,101.0,250.0,2026-01-10
1,1,Aysel,Baku,102.0,400.0,2026-02-05
2,1,Aysel,Baku,106.0,350.0,2026-03-20
3,2,Orxan,Baku,103.0,150.0,2026-02-20
4,3,Leyla,Baku,104.0,900.0,2026-03-01
5,3,Leyla,Baku,105.0,300.0,2026-03-15
6,4,Murad,Baku,NaN,NaN,NaT


In [17]:
orders.sort_values("amount",ascending=0)

,order_id,customer_id,amount,order_date
3,104,3,900,2026-03-01
1,102,1,400,2026-02-05
5,106,1,350,2026-03-20
4,105,3,300,2026-03-15
0,101,1,250,2026-01-10
2,103,2,150,2026-02-20


In [40]:
#Tapşırıq 1 (asan): Məbləği 300-dən çox olan sifarişləri göstər. Yalnız order_id, amount sütunları çıxsın.
new_orders=orders[orders["amount"]>300]
new_orders[["order_id","amount"]]

,order_id,amount
1,102,400
3,104,900
5,106,350


In [28]:
#Hər müştərinin adını və ümumi sifariş məbləğini göstər (name, total_amount).
# İpucu: əvvəl merge, sonra groupby("name"). Sütunun adını dəyişmək üçün .rename(columns={"amount": "total_amount"}) işlət.
customers.merge(orders, on='customer_id').groupby("name")["amount"].sum().rename({"amount":"ll"})

,amount
name,
Aysel,1000
Leyla,1200
Orxan,150


In [30]:
#Heç sifarış verməmiş müşteriləri tap
merged = customers.merge(orders, on="customer_id", how="left")
merged[merged['order_id'].isna()]

,customer_id,name,city,order_id,amount,order_date
6,4,Murad,Baku,NaN,NaN,NaT


In [48]:
#runnig total
orders['running_total']=orders.sort_values('order_date').groupby("customer_id")["amount"].cumsum()
orders

,order_id,customer_id,amount,order_date,running_total
0,101,1,250,2026-01-10,250
1,102,1,400,2026-02-05,650
2,103,2,150,2026-02-20,150
3,104,3,900,2026-03-01,900
4,105,3,300,2026-03-15,1200
5,106,1,350,2026-03-20,1000


In [56]:
orders.sort_values(["customer_id","order_date"])
orders['prev_date']=orders.groupby("customer_id")["order_date"].shift(1)
orders["prev_day_diff"]=(orders['order_date']-orders['prev_date']).dt.days
orders

,order_id,customer_id,amount,order_date,running_total,prev_date,prev_day_diff
0,101,1,250,2026-01-10,250,NaT,NaN
1,102,1,400,2026-02-05,650,2026-01-10,26.0
2,103,2,150,2026-02-20,150,NaT,NaN
3,104,3,900,2026-03-01,900,NaT,NaN
4,105,3,300,2026-03-15,1200,2026-03-01,14.0
5,106,1,350,2026-03-20,1000,2026-02-05,43.0


In [47]:
orders.drop_duplicates(subset="customer_id",keep="last")

,order_id,customer_id,amount,order_date
2,103,2,150,2026-02-20
4,105,3,300,2026-03-15
5,106,1,350,2026-03-20
